# Phase 1: Data Loading & Inspection

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

# 1. Load the data 
# Using "../" goes up one directory level to find the "data" folder
file_path = "../data/risk_factors_cervical_cancer.csv"

df = pd.read_csv(file_path)

# 2. Check the shape
rows, cols = df.shape
print(f"Dataset Shape: {rows} patients and {cols} features\n")

# Peek at the first few rows
display(df.head())

Dataset Shape: 858 patients and 36 features



,Age,Number of sexual partners,First sexual intercourse,Num of pregnancies,Smokes,Smokes (years),Smokes (packs/year),Hormonal Contraceptives,Hormonal Contraceptives (years),IUD,...,STDs: Time since first diagnosis,STDs: Time since last diagnosis,Dx:Cancer,Dx:CIN,Dx:HPV,Dx,Hinselmann,Schiller,Citology,Biopsy
0,18,4.0,15.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
1,15,1.0,14.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
2,34,1.0,?,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
3,52,5.0,16.0,4.0,1.0,37.0,37.0,1.0,3.0,0.0,...,?,?,1,0,1,0,0,0,0,0
4,46,3.0,21.0,4.0,0.0,0.0,0.0,1.0,15.0,0.0,...,?,?,0,0,0,0,0,0,0,0


### Initial Diagnostic Checks

In [2]:
# 3. Inspect Data Types
print("--- DataFrame Info ---")
df.info()
print("\n")

# 4. Investigate Suspicious 'Object' Columns
suspicious_col = 'Number of sexual partners' 
print(f"--- Unique values in '{suspicious_col}' ---")
print(df[suspicious_col].unique()) 
print("\n")

# 5. Identify the Target Variable
target = 'Biopsy'
print(f"--- Target Variable Distribution ({target}) ---")
display(df[target].value_counts(normalize=True).round(4) * 100)

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 858 entries, 0 to 857
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   Age                                 858 non-null    int64 
 1   Number of sexual partners           858 non-null    object
 2   First sexual intercourse            858 non-null    object
 3   Num of pregnancies                  858 non-null    object
 4   Smokes                              858 non-null    object
 5   Smokes (years)                      858 non-null    object
 6   Smokes (packs/year)                 858 non-null    object
 7   Hormonal Contraceptives             858 non-null    object
 8   Hormonal Contraceptives (years)     858 non-null    object
 9   IUD                                 858 non-null    object
 10  IUD (years)                         858 non-null    object
 11  STDs                               

Biopsy
0    93.59
1     6.41
Name: proportion, dtype: float64

# Phase 2: Data Cleaning & Type Conversion

In [3]:
# 1. Replace the text '?' with standard numpy NaN
df = df.replace('?', np.nan)

# 2. Convert all columns to numeric (Future-Proof Version)
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except ValueError:
        pass

print("Data types successfully converted!\n")

print("--- Updated DataFrame Info ---")
df.info()

Data types successfully converted!

--- Updated DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 858 entries, 0 to 857
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Age                                 858 non-null    int64  
 1   Number of sexual partners           832 non-null    float64
 2   First sexual intercourse            851 non-null    float64
 3   Num of pregnancies                  802 non-null    float64
 4   Smokes                              845 non-null    float64
 5   Smokes (years)                      845 non-null    float64
 6   Smokes (packs/year)                 845 non-null    float64
 7   Hormonal Contraceptives             750 non-null    float64
 8   Hormonal Contraceptives (years)     750 non-null    float64
 9   IUD                                 741 non-null    float64
 10  IUD (years)                         741 non

# Phase 3: Missing Data Assessment & Investigation

In [4]:
# Calculate the percentage of missing values per column
missing_percentages = (df.isnull().sum() / len(df)) * 100

# Filter to only show columns that actually have missing data, and sort them
missing_columns = missing_percentages[missing_percentages > 0].sort_values(ascending=False)

print("Percentage of missing data per column:")
display(missing_columns.round(2).astype(str) + ' %')

Percentage of missing data per column:


STDs: Time since first diagnosis      91.72 %
STDs: Time since last diagnosis       91.72 %
IUD (years)                           13.64 %
IUD                                   13.64 %
Hormonal Contraceptives               12.59 %
Hormonal Contraceptives (years)       12.59 %
STDs (number)                         12.24 %
STDs                                  12.24 %
STDs:vulvo-perineal condylomatosis    12.24 %
STDs:vaginal condylomatosis           12.24 %
STDs:pelvic inflammatory disease      12.24 %
STDs:syphilis                         12.24 %
STDs:genital herpes                   12.24 %
STDs:molluscum contagiosum            12.24 %
STDs:HIV                              12.24 %
STDs:AIDS                             12.24 %
STDs:condylomatosis                   12.24 %
STDs:cervical condylomatosis          12.24 %
STDs:HPV                              12.24 %
STDs:Hepatitis B                      12.24 %
Num of pregnancies                     6.53 %
Number of sexual partners         

In [5]:
# 1. Identify the "Worst Offenders" (Columns missing > 50% of data)
worst_offenders = missing_percentages[missing_percentages > 50].index.tolist()
print(f"Columns missing > 50% of data:\n{worst_offenders}\n")

# 2. Look at the surviving data
print("--- Summary of the SURVIVING data in these columns ---")
display(df[worst_offenders].describe().round(2))

# 3. Test a Medical Hypothesis
# If a patient never had an STD, the "Time since first diagnosis" will be blank.
if 'STDs: Time since first diagnosis' in worst_offenders:
    print("\n--- Testing the Missing Data Hypothesis ---")
    
    # Create a filter (mask) for rows where the time is missing
    missing_time_mask = df['STDs: Time since first diagnosis'].isnull()
    
    print("For patients missing 'Time since first diagnosis', how many STDs did they have?")
    display(df.loc[missing_time_mask, 'STDs (number)'].value_counts(dropna=False))

Columns missing > 50% of data:
['STDs: Time since first diagnosis', 'STDs: Time since last diagnosis']

--- Summary of the SURVIVING data in these columns ---


,STDs: Time since first diagnosis,STDs: Time since last diagnosis
count,71.00,71.00
mean,6.14,5.82
std,5.90,5.76
min,1.00,1.00
25%,2.00,2.00
50%,4.00,3.00
75%,8.00,7.50
max,22.00,22.00



--- Testing the Missing Data Hypothesis ---
For patients missing 'Time since first diagnosis', how many STDs did they have?


STDs (number)
0.0    674
NaN    105
2.0      4
1.0      4
Name: count, dtype: int64

# Phase 4: Handling Missing Data (Imputation)

In [6]:
# 1. Drop the ruined columns
columns_to_drop = ['STDs: Time since first diagnosis', 'STDs: Time since last diagnosis']
df = df.drop(columns=columns_to_drop, errors='ignore') 
print(f"Dropped {len(columns_to_drop)} columns.\n")

# 2. Impute the remaining columns using the Median
print("--- Imputed Median Values ---")
for col in df.columns:
    # Check if the column has ANY missing data
    if df[col].isnull().sum() > 0:
        median_value = df[col].median()
        print(f"{col}: {median_value}")
        
        # Fill the NaN values
        df[col] = df[col].fillna(median_value)

# 3. Final Proof
print("\n--- Final Dataset Check ---")
print(f"Total missing values left in the ENTIRE dataset: {df.isnull().sum().sum()}")
print(f"Final Dataset Shape: {df.shape[0]} patients, {df.shape[1]} features")

Dropped 2 columns.

--- Imputed Median Values ---
Number of sexual partners: 2.0
First sexual intercourse: 17.0
Num of pregnancies: 2.0
Smokes: 0.0
Smokes (years): 0.0
Smokes (packs/year): 0.0
Hormonal Contraceptives: 1.0
Hormonal Contraceptives (years): 0.5
IUD: 0.0
IUD (years): 0.0
STDs: 0.0
STDs (number): 0.0
STDs:condylomatosis: 0.0
STDs:cervical condylomatosis: 0.0
STDs:vaginal condylomatosis: 0.0
STDs:vulvo-perineal condylomatosis: 0.0
STDs:syphilis: 0.0
STDs:pelvic inflammatory disease: 0.0
STDs:genital herpes: 0.0
STDs:molluscum contagiosum: 0.0
STDs:AIDS: 0.0
STDs:HIV: 0.0
STDs:Hepatitis B: 0.0
STDs:HPV: 0.0

--- Final Dataset Check ---
Total missing values left in the ENTIRE dataset: 0
Final Dataset Shape: 858 patients, 34 features


In [7]:
# Phase 5: Export Clean Data
# Save the cleaned dataset so we can pick it up in the next notebook
output_path = "../data/clean_cervical_cancer_data.csv"
df.to_csv(output_path, index=False)

print(f"Clean data successfully saved to {output_path}")

Clean data successfully saved to ../data/clean_cervical_cancer_data.csv
